## Feature Engineering, Sistema de Predicción Energética y Alerta Temprana en Barrios de Barcelona

> **Contexto:** A partir del dataset limpio producido en el EDA (424.368 registros x 30 columnas,
> 2019-2025, bloques de 6h, 42 codigos postales), el objetivo de esta fase es construir
> las variables derivadas que alimentaran los modelos de prediccion. Las decisiones de
> feature engineering estan fundamentadas en los hallazgos del EDA: patrones temporales
> confirmados por STL y ACF/PACF, relaciones no lineales con variables climaticas, y
> heterogeneidad espacial por codigo postal.

### Variables a construir

> Lags temporales

- lag_1 (6h antes): contribucion directa confirmada por PACF (0.37).
- lag_4 (24h antes): el mas importante, PACF 0.75. Mismo bloque del dia anterior.
- lag_28 (7 dias antes): ACF 0.87, captura el patron semanal completo.
- rolling_mean_7d: promedio de los ultimos 7 dias, captura tendencia reciente.
- lag_56 y lag_84 (2 y 3 semanas): correlacion existe pero decrece. lag_28
  ya captura el patron semanal. Se evaluan tras modelado via SHAP.

> Variables de interaccion

- hora_x_finde = hora * es_finde: captura que el perfil horario del domingo
  es cualitativamente distinto al del miercoles. Solo XGBoost y LightGBM.
- is_covid = 1 si anio == 2020: controla el cambio estructural por pandemia.

> Variables climaticas derivadas

- HDD = max(0, 15 - temp_mean): grados de calefaccion [Kuru & Calis 2019].
- CDD = max(0, temp_mean - 22): grados de refrigeracion [Kuru & Calis 2019].
- lst_nublado = 1 si lst_celsius es nulo: flag de fiabilidad de MODIS.
- precipitacion_llueve = 1 si precipitacion_sum > 0: binarizacion confirmada
  en EDA (ambas medianas en 0, informacion util solo en si llueve o no).

> Imputacion meteorologica X2

- temp X2 2025: imputar desde X4 con factor de correccion historico
  ratio X2/X4 (2019-2023).
- viento e irradiancia X2: siempre desde X4 (X2 nunca tuvo sensores).
- lst_celsius sin cobertura: MODIS como respaldo espacial.
- irradiancia_mean: topar a 0 los valores negativos de sensor.

> Vecinos geograficos (deprioritizado)

- lag1_vecinos: consumo promedio de CPs colindantes en t-1, calculado con
  shapely sobre BARCELONA.geojson. Se descarta si hay restriccion de tiempo
  y se documenta como linea futura.

### Variables del dataset de entrada

| Variable | Tipo | Descripcion |
|---|---|---|
| `cod_postal` | Categorica | Codigo postal de Barcelona (08001-08042) |
| `nombre_postal` | Categorica | Nombre del barrio asociado al codigo postal |
| `datetime` | Temporal | Inicio del bloque de 6 horas |
| `mwh_total` | Target | Consumo electrico total en MWh |
| `lst_celsius` | Numerica | Land Surface Temperature — MODIS (°C) |
| `temp_mean` | Numerica | Temperatura media Meteocat (°C) |
| `humedad_mean` | Numerica | Humedad relativa media (%) |
| `viento_mean` | Numerica | Velocidad media del viento (m/s) |
| `precipitacion_sum` | Numerica | Precipitacion acumulada en el bloque (mm) |
| `irradiancia_mean` | Numerica | Irradiancia solar global media (W/m²) |
| `es_festivo` | Binaria | Es dia festivo |
| `hora` | Ordinal | Hora de inicio del bloque (0, 6, 12, 18) |
| `dia_semana` | Ordinal | Dia de la semana (0=Lunes, 6=Domingo) |
| `mes` | Ordinal | Mes del ano (1-12) |
| `anio` | Numerica | Ano |
| `es_finde` | Binaria | Es fin de semana |

## Librerias

In [1]:
import polars as pl
from pymongo import MongoClient

In [2]:
client = MongoClient('mongodb://mongo:27017/')
db     = client['tfm_energy']

docs = list(db['dataset_clean'].find({}, {'_id': 0}))
df   = pl.DataFrame(docs, infer_schema_length=None)

print(f"Shape: {df.shape}")
print(f"Desde: {df['datetime'].min()}")
print(f"Hasta: {df['datetime'].max()}")
print(f"Nulos mwh_total: {df['mwh_total'].null_count()}")
print(f"Codigos postales: {df['cod_postal'].n_unique()}")

Shape: (424368, 30)
Desde: 2019-01-01 00:00:00
Hasta: 2025-11-30 18:00:00
Nulos mwh_total: 0
Codigos postales: 42
